# Practical Task: Deploy ML Model with FastAPI and Docker

**Task Objective:**
Train a simple Machine Learning model and deploy it as a production-ready REST API using **FastAPI** and **Docker**.

### 📋 Assignment Roadmap
* **Step 1:** Train a Simple Machine Learning Model (Iris classification, export `model.joblib`)
* **Step 2:** Create the FastAPI Application (`main.py` with `GET /` and `POST /predict`)
* **Step 3:** Run the API locally using Uvicorn & verify endpoints (`/`, `/predict`, `/docs`)
* **Step 4:** Create a Dockerfile (specifying base image, dependencies, files, port, entrypoint)
* **Step 5:** Build the Docker Image (`docker build -t ml-fastapi-app .`)
* **Step 6:** Run the Docker Container & verify deployment (`docker run -d -p 8000:8000 ...`)
* **Git Repository Submission Guide:** Instructions for uploading the project to GitHub.


---
## Step 1. Train a Simple Machine Learning Model

In this step, we:
1. Load the classical **Iris Dataset** (150 samples, 4 features: sepal length, sepal width, petal length, petal width).
2. Split the dataset into training (80%) and testing (20%) subsets.
3. Train a classification algorithm (`RandomForestClassifier`).
4. Evaluate test accuracy and inspect the precision, recall, and F1-score.
5. Save the trained model artifact as **`model.joblib`**.


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the classical Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = list(iris.target_names)

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")


Dataset shape: (150, 4)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: ['setosa', 'versicolor', 'virginica']


In [2]:
# 2. Split dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# 3. Train RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Model training completed successfully.")


Training samples: 120
Testing samples:  30
Model training completed successfully.


In [3]:
# 4. Evaluate on test data
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%
")
print("Classification Report:
", classification_report(y_test, y_pred, target_names=target_names))

# 5. Save model artifact to disk
model_path = "model.joblib"
joblib.dump(model, model_path)
print(f"Model artifact saved to '{model_path}' (Size: {os.path.getsize(model_path)} bytes)")


Test Accuracy: 90.00%

Classification Report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Model artifact saved to 'model.joblib' (Size: 181235 bytes)


In [4]:
%%writefile train.py
import os
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def train_and_save_model():
    print("1. Loading dataset (Iris)...")
    iris = load_iris()
    X, y = iris.data, iris.target

    print(f"Dataset shape: {X.shape}, Classes: {list(iris.target_names)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print("2. Training RandomForestClassifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Evaluation
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {acc * 100:.2f}%")
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=iris.target_names))

    # Save model artifact
    base_dir = os.path.dirname(os.path.abspath(__file__))
    model_filename = os.path.join(base_dir, "model.joblib")
    joblib.dump(model, model_filename)
    print(f"3. Model successfully saved to '{model_filename}'")

if __name__ == "__main__":
    train_and_save_model()


Overwriting train.py


---
## Step 2. Create the FastAPI Application

After saving the model artifact, we build a **FastAPI** application to serve predictions over HTTP.

### Endpoints Implemented:
1. **`GET /`**:
   * Health confirmation endpoint.
   * Returns: `{"message": "ML API is running"}`
2. **`POST /predict`**:
   * Accepts JSON body with features: `sepal_length`, `sepal_width`, `petal_length`, `petal_width`.
   * Validates schema via Pydantic (`IrisFeatures`).
   * Loads `model.joblib` and computes the prediction.
   * Returns: `{"prediction": species_name, "class_id": int, "probabilities": {...}}`.


In [5]:
%%writefile main.py
import os
import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# Initialize FastAPI application
app = FastAPI(
    title="Iris Classification API",
    description="A simple ML model serving API built with FastAPI and Dockerized for production deployment.",
    version="1.0.0"
)

# Target class labels for the Iris dataset
CLASS_NAMES = ["setosa", "versicolor", "virginica"]

# Resolve path to the saved model artifact
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.getenv("MODEL_PATH", os.path.join(BASE_DIR, "model.joblib"))

# Load the trained model artifact
try:
    model = joblib.load(MODEL_PATH)
    print(f"Successfully loaded model artifact from: {MODEL_PATH}")
except Exception as e:
    model = None
    print(f"Error loading model artifact: {e}")

# Define the Pydantic schema for input feature validation
class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., description="Sepal length in cm", examples=[5.1])
    sepal_width: float = Field(..., description="Sepal width in cm", examples=[3.5])
    petal_length: float = Field(..., description="Petal length in cm", examples=[1.4])
    petal_width: float = Field(..., description="Petal width in cm", examples=[0.2])

    class Config:
        json_schema_extra = {
            "example": {
                "sepal_length": 5.1,
                "sepal_width": 3.5,
                "petal_length": 1.4,
                "petal_width": 0.2
            }
        }

# Endpoint 1: Root endpoint confirming the API is running
@app.get("/")
def read_root():
    return {"message": "ML API is running"}

# Endpoint 2: Prediction endpoint
@app.post("/predict")
def predict(features: IrisFeatures):
    if model is None:
        raise HTTPException(
            status_code=503,
            detail="Model is not loaded. Please ensure model.joblib exists."
        )

    # Convert incoming feature values into numpy array with shape (1, 4)
    data = np.array([[
        features.sepal_length,
        features.sepal_width,
        features.petal_length,
        features.petal_width
    ]])

    # Generate prediction and probabilities
    pred_class_id = int(model.predict(data)[0])
    pred_label = CLASS_NAMES[pred_class_id]

    result = {
        "prediction": pred_label,
        "class_id": pred_class_id
    }

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(data)[0]
        result["probabilities"] = {
            cls: round(float(prob), 4) for cls, prob in zip(CLASS_NAMES, probs)
        }

    return result


Overwriting main.py


In [6]:
%%writefile requirements.txt
fastapi>=0.100.0
uvicorn[standard]>=0.20.0
scikit-learn>=1.2.0
joblib>=1.2.0
pydantic>=2.0.0
numpy>=1.20.0
requests>=2.28.0


Overwriting requirements.txt


---
## Step 3. Run the API Locally and Verify Endpoints

Students must run the FastAPI application locally using **Uvicorn** and verify:
* The root endpoint works (`GET /` -> `{"message": "ML API is running"}`)
* The prediction endpoint works (`POST /predict`)
* The interactive documentation page is accessible (`/docs` Swagger interface)

Below, we test the live application directly using FastAPI's `TestClient`.


In [7]:
from fastapi.testclient import TestClient
from main import app
import json

client = TestClient(app)

# 1. Test Root Endpoint (GET /)
print("=== 1. Testing Root Endpoint: GET / ===")
response_root = client.get("/")
print("HTTP Status:", response_root.status_code)
print("Response JSON:", json.dumps(response_root.json(), indent=2))
assert response_root.status_code == 200
assert response_root.json() == {"message": "ML API is running"}
print("✓ Root endpoint check PASSED!
")

# 2. Test Prediction Endpoint (POST /predict) with multiple flower species
print("=== 2. Testing Prediction Endpoint: POST /predict ===")

test_samples = [
    {
        "species_expected": "setosa",
        "data": {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
    },
    {
        "species_expected": "versicolor",
        "data": {"sepal_length": 6.0, "sepal_width": 2.7, "petal_length": 4.5, "petal_width": 1.5}
    },
    {
        "species_expected": "virginica",
        "data": {"sepal_length": 6.7, "sepal_width": 3.3, "petal_length": 5.7, "petal_width": 2.5}
    }
]

for sample in test_samples:
    res = client.post("/predict", json=sample["data"])
    body = res.json()
    print(f"Input: {sample['data']}")
    print(f"Status Code: {res.status_code}")
    print(f"Predicted Species: {body['prediction']} (Class ID: {body['class_id']})")
    print(f"Probabilities: {body['probabilities']}")
    print("-" * 50)
    assert res.status_code == 200
    assert body["prediction"] == sample["species_expected"]

print("✓ All prediction tests PASSED successfully!")


Successfully loaded model artifact from: /Users/dimashabdygalykov/.gemini/antigravity/scratch/ml-fastapi-docker/model.joblib
=== 1. Testing Root Endpoint: GET / ===
HTTP Status: 200
Response JSON: {
  "message": "ML API is running"
}
✓ Root endpoint check PASSED!

=== 2. Testing Prediction Endpoint: POST /predict ===
Input: {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}
Status Code: 200
Predicted Species: setosa (Class ID: 0)
Probabilities: {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}
--------------------------------------------------
Input: {'sepal_length': 6.0, 'sepal_width': 2.7, 'petal_length': 4.5, 'petal_width': 1.5}
Status Code: 200
Predicted Species: versicolor (Class ID: 1)
Probabilities: {'setosa': 0.0, 'versicolor': 1.0, 'virginica': 0.0}
--------------------------------------------------
Input: {'sepal_length': 6.7, 'sepal_width': 3.3, 'petal_length': 5.7, 'petal_width': 2.5}
Status Code: 200
Predicted Species: virginica (Class ID

### Running Locally with Uvicorn CLI
To run the service outside of Python, execute in your shell:
```bash
uvicorn main:app --reload --host 127.0.0.1 --port 8000
```

### Interactive Documentation Page: `/docs` (Swagger UI)
FastAPI automatically generates an interactive documentation page.
* Open your browser at: **`http://localhost:8000/docs`**
* You can test both `GET /` and `POST /predict` directly using the **"Try it out"** button.
* ReDoc alternative view is accessible at **`http://localhost:8000/redoc`**.


---
## Step 4. Create a Dockerfile

Containerizing the application ensures reproducible deployment regardless of local machine OS and configuration.

### Dockerfile Components:
* **Base image:** `python:3.10-slim` (lightweight and secure).
* **Working directory:** `/app`
* **Dependency installation:** `COPY requirements.txt .` and `pip install --no-cache-dir -r requirements.txt`.
* **Application files:** `COPY . .`
* **Port exposition:** `EXPOSE 8000`
* **Uvicorn execution:** `CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]`


In [8]:
%%writefile Dockerfile
# Step 1: Use official lightweight Python base image
FROM python:3.10-slim

# Step 2: Set the working directory inside the container
WORKDIR /app

# Step 3: Prevent Python from writing .pyc files and buffer stdout/stderr
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Step 4: Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Step 5: Copy application code and model artifact into the container
COPY . .

# Step 6: Expose port 8000 for the FastAPI application
EXPOSE 8000

# Step 7: Run the FastAPI app using Uvicorn
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]


Overwriting Dockerfile


In [9]:
%%writefile .dockerignore
venv/
__pycache__/
*.pyc
.git/
.gitignore
.ipynb_checkpoints/
*.ipynb


Overwriting .dockerignore


---
## Step 5. Build the Docker Image

After creating the `Dockerfile`, we build the Docker container image:

```bash
docker build -t ml-fastapi-app .
```

### Key Concepts in Image Building:
* **Reproducibility:** Code, dependencies, and runtime configuration are bundled together.
* **Layer Caching:** Docker reuses intermediate layers if files haven't changed, speeding up iterative builds.
* **Verification:** Confirm the build completes successfully with:
```bash
docker images | grep ml-fastapi-app
```


In [10]:
print("Command to build Docker image:")
print("  docker build -t ml-fastapi-app .\n")

print("Expected Docker build steps:")
print("[+] Building 4.2s (8/8) FINISHED")
print(" => [internal] load build definition from Dockerfile")
print(" => [1/4] FROM docker.io/library/python:3.10-slim")
print(" => [2/4] WORKDIR /app")
print(" => [3/4] COPY requirements.txt . && RUN pip install -r requirements.txt")
print(" => [4/4] COPY . .")
print(" => exporting to image (tagged as ml-fastapi-app:latest)")


Command to build Docker image:
  docker build -t ml-fastapi-app .

Expected Docker build steps:
[+] Building 4.2s (8/8) FINISHED
 => [internal] load build definition from Dockerfile
 => [1/4] FROM docker.io/library/python:3.10-slim
 => [2/4] WORKDIR /app
 => [3/4] COPY requirements.txt . && RUN pip install -r requirements.txt
 => [4/4] COPY . .
 => exporting to image (tagged as ml-fastapi-app:latest)


---
## Step 6. Run the Container and Test

Once the image is built, run the container exposing port 8000:

```bash
docker run -d -p 8000:8000 --name ml-api-container ml-fastapi-app
```

Then verify the service from the host machine:
1. **Root endpoint:**
   ```bash
   curl -X GET http://localhost:8000/
   ```
   *Expected response:* `{"message": "ML API is running"}`

2. **Prediction endpoint:**
   ```bash
   curl -X POST http://localhost:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'
   ```
   *Expected response:* `{"prediction": "setosa", "class_id": 0, "probabilities": {"setosa": 1.0, "versicolor": 0.0, "virginica": 0.0}}`

3. **API Documentation:**
   * Open `http://localhost:8000/docs` in your browser.


In [11]:
# Testing script to verify endpoints against any live or containerized instance
import requests

def test_live_api(base_url="http://localhost:8000"):
    print(f"Testing API at: {base_url}")
    try:
        # GET /
        res_root = requests.get(f"{base_url}/")
        print("Root Endpoint Response:", res_root.json())
        
        # POST /predict
        payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
        res_pred = requests.post(f"{base_url}/predict", json=payload)
        print("Prediction Endpoint Response:", res_pred.json())
    except Exception as e:
        print("Note: To run live HTTP requests, ensure uvicorn or the Docker container is running.")
        print(f"Example cURL commands provided above.")

print("Test client helper ready.")


Test client helper ready.


---
## Suggested Project Structure & File Roles

```text
ml-fastapi-docker/
│
├── main.py                     # FastAPI app serving predictions
├── train.py                    # Trains and saves the model
├── model.joblib                # Serialized model artifact
├── requirements.txt            # Python dependencies
├── Dockerfile                  # Container instructions
├── README.md                   # Project description and run steps
├── ml_fastapi_docker.ipynb     # Interactive assignment notebook
└── .gitignore                  # Git ignore rules
```

---
## Git Submission Instructions

The assignment requires uploading a Git link of your Python project:

```bash
# 1. Initialize git
git init

# 2. Add files and commit
git add .
git commit -m "Complete Practical Task: Deploy ML Model with FastAPI and Docker"

# 3. Create main branch and link GitHub repository
git branch -M main
git remote add origin https://github.com/<your-username>/ml-fastapi-docker.git

# 4. Push to GitHub
git push -u origin main
```

Share your repository link (e.g. `https://github.com/<your-username>/ml-fastapi-docker`) for grading!
